# Step 05 — Traffic Quality Gate & Corridor Probe Planning


In [5]:
import sys
import pandas as pd
from IPython.display import display

from routing.config import ORIGIN, ORIGINS, DESTINATION
from routing.map_loader import load_road_graph
from routing.map_view import interactive_map
from routing.traffic_api import TomTomClient
from routing.traffic_pipeline import collect_traffic_snapshot, make_routing_graph, save_summary
from routing.traffic_quality import plan_corridor_probes, audit_traffic_snapshot

print("Kernel:", sys.executable)


Kernel: /Users/mojtabashabani/wiser/mary/.venv/bin/python


In [4]:
assert len(plan) > 0, "No valid probes were planned"

probe = plan[0]
probe_edge = probe["edge_id"]

print("Selected edge:", probe_edge)
print("Latitude:", probe["latitude"])
print("Longitude:", probe["longitude"])
print("Road class:", probe["road_class"])


# STEP 2: Make exactly ONE TomTom request

with TomTomClient() as client:

    test_snapshot, test_summary = collect_traffic_snapshot(
        G,
        client,
        max_requests=1,
        selected_edge_ids=[probe_edge],
        pause_s=0,
    )


# STEP 3: Inspect the result

u, v, k = probe_edge
edge = test_snapshot[u][v][k]

print("\n--- API DIAGNOSTICS ---")

print("Requests:", test_summary["requests_made"])
print("Request errors:", test_summary["request_errors"])

print(
    "API errors:",
    test_summary["api_error_groups"]
)

print(
    "Matching diagnostics:",
    test_summary["map_matching_diagnostics"]
)

print("\n--- EDGE RESULT ---")

print("Traffic status:", edge["traffic_status"])
print("Match reason:", edge["traffic_match_reason"])
print("Traffic reason:", edge["traffic_reason"])

print("Confidence:", edge["traffic_confidence"])
print("Current speed:", edge["current_speed_kph"])
print("Free-flow speed:", edge["free_flow_speed_kph"])

print("Travel time:", edge["travel_time_s"])
print("Imputed:", edge["time_is_imputed"])


NameError: name 'plan' is not defined

## ۱. بارگذاری گراف OSM (بدون TomTom)


In [6]:
RADIUS_M = 2000

G = load_road_graph(radius_m=RADIUS_M)

print("Graph:", type(G).__name__)
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

assert G.is_directed()
assert G.is_multigraph()

print("PASS: OSM Graph")

Graph: MultiDiGraph
Nodes: 2569
Edges: 5850
PASS: OSM Graph


## ۲. برنامه‌ریزی Probeها بدون API


In [7]:
REQUEST_LIMIT = 12

plan = plan_corridor_probes(
    G,
    max_probes=REQUEST_LIMIT,
    origins=ORIGINS,
    destination=DESTINATION,
    corridor_width_m=350,
    min_length_m=35,
    min_spacing_m=65,
)

print("Planned requests:", len(plan))

display(pd.DataFrame(plan))

Planned requests: 12


,edge_id,latitude,longitude,length_m,road_class,corridor,corridor_offset_m
0,"(109869, 265563774, 0)",51.495550,-0.181906,134.42,residential,Gloucester Road,1.3
1,"(26389375, 34519894, 0)",51.498980,-0.174206,181.39,unclassified,Royal Albert Hall,2.2
2,"(1360237602, 3925964624, 0)",51.495273,-0.174187,68.82,residential,South Kensington,0.9
3,"(1027237558, 1116949690, 0)",51.496115,-0.180221,187.61,residential,Gloucester Road,18.2
4,"(1691189781, 20821126, 0)",51.500769,-0.176459,72.93,residential,Royal Albert Hall,45.6
5,"(34519894, 6280192964, 0)",51.496979,-0.173807,266.83,unclassified,South Kensington,27.3
6,"(1356901661, 109869, 0)",51.495093,-0.182533,41.83,trunk,Gloucester Road,23.2
7,"(20947178, 29353841, 0)",51.500597,-0.178215,200.15,residential,Royal Albert Hall,49.2
8,"(26389442, 34519894, 0)",51.498196,-0.173748,41.05,residential,South Kensington,31.4
9,"(1027237558, 860183693, 0)",51.496988,-0.179934,67.30,residential,Gloucester Road,49.3


## ۳. اجرای کنترل‌شده جمع‌آوری


In [8]:
RUN_LIVE_TRAFFIC = False

selected_edges = [
    row["edge_id"]
    for row in plan
]

if RUN_LIVE_TRAFFIC:
    with TomTomClient() as client:
        snapshot, summary = collect_traffic_snapshot(
            G,
            client,
            max_requests=REQUEST_LIMIT,
            selected_edge_ids=selected_edges,
        )

    save_summary(summary)

else:
    snapshot, summary = collect_traffic_snapshot(
        G,
        None,
        max_requests=0,
        selected_edge_ids=selected_edges,
    )

print("Requests:", summary["requests_made"])
print("Counts:", summary["counts"])

Requests: 0
Counts: {'missing': 5850, 'red': 0, 'yellow': 0, 'clear': 0, 'closed': 0}


In [9]:
RUN_LIVE_TRAFFIC = True
REQUEST_LIMIT = 1

selected_edges = [
    (1356901661, 109869, 0)
]

assert G.has_edge(*selected_edges[0])

with TomTomClient() as client:
    snapshot, summary = collect_traffic_snapshot(
        G,
        client,
        max_requests=REQUEST_LIMIT,
        selected_edge_ids=selected_edges,
        pause_s=0.2,
    )

save_summary(summary)

print("========== TOMTOM SUMMARY ==========")
print("Requests:", summary["requests_made"])
print("Errors:", summary["request_errors"])
print("API errors:", summary["api_error_groups"])
print("Matching:", summary["map_matching_diagnostics"])

u, v, k = selected_edges[0]
edge = snapshot[u][v][k]

print("\n========== EDGE RESULT ==========")
print("Matched:", edge["traffic_matched"])
print("Match reason:", edge["traffic_match_reason"])
print("Traffic status:", edge["traffic_status"])
print("Traffic reason:", edge["traffic_reason"])
print("Current speed:", edge["current_speed_kph"])
print("Free flow speed:", edge["free_flow_speed_kph"])
print("Confidence:", edge["traffic_confidence"])
print("Travel time:", edge["travel_time_s"])
print("Imputed:", edge["time_is_imputed"])


========== TOMTOM SUMMARY ==========
Requests: 1
Errors: 0
API errors: []
Matching: {'candidate_match': 1}

========== EDGE RESULT ==========
Matched: True
Match reason: candidate_match
Traffic status: clear
Traffic reason: observed
Current speed: 27.0
Free flow speed: 27.0
Confidence: 1.0
Travel time: 5.577524021559747
Imputed: False


## ۴. Audit کیفیت و قابلیت آماده‌سازی وزن‌ها



In [10]:
report = audit_traffic_snapshot(
    G,
    snapshot,
    summary
)

print(
    "Graph identity:",
    report["graph_identity_ok"]
)

print(
    "Costs valid:",
    report["costs_ok"]
)

print(
    "Algorithm smoke test ready:",
    report["algorithm_smoke_test_ready"]
)

print(
    "Final traffic comparison ready:",
    report["final_traffic_comparison_ready"]
)

print("Warnings:", report["warnings"])

Graph identity: True
Costs valid: True
Algorithm smoke test ready: True
Final traffic comparison ready: False
Warnings: ['Observed traffic coverage is sparse; most travel times are not measured', 'TomTom geometry point order is not authoritative legal direction proof']


## ۵. نقشه تعاملی و لایه‌های ترافیک


In [11]:
traffic_widget, ui_state = interactive_map(
    G,
    origin=ORIGIN,
    destination=DESTINATION,
    traffic_graph=snapshot
)

display(traffic_widget)

In [13]:
print("Origin:", ui_state["origin_node"])
print("Destination:", ui_state["destination_node"])
print("Last click:", ui_state["last_click_latlon"])

if ui_state["last_click_latlon"] is not None:
    assert ui_state["origin_node"] in G
    assert ui_state["destination_node"] in G
    print("PASS: Interactive Map")
else:
    print("No double-click received yet")

Origin: 941233614
Destination: 26389375
Last click: (51.49831428250405, -0.18459073078619384)
PASS: Interactive Map


## ۶. سه سناریو، دو Objective، بدون اجرای الگوریتم‌ها



In [14]:
free_flow_time = make_routing_graph(
    snapshot,
    objective="time",
    scenario="free_flow"
)

measured_time = make_routing_graph(
    snapshot,
    objective="time",
    scenario="measured"
)

measured_distance = make_routing_graph(
    snapshot,
    objective="distance",
    scenario="measured"
)

print(
    "Free flow:",
    free_flow_time.number_of_edges()
)

print(
    "Measured time:",
    measured_time.number_of_edges()
)

print(
    "Measured distance:",
    measured_distance.number_of_edges()
)

print("PASS: Routing graphs prepared") 

Free flow: 5850
Measured time: 5850
Measured distance: 5850
PASS: Routing graphs prepared


In [15]:
from routing.traffic_api import TomTomClient
from routing.traffic_pipeline import collect_traffic_snapshot

# Three controlled probes
selected_edges = [
    (1356901661, 109869, 0),
    (26389375, 34519894, 0),
    (7902276163, 25291778, 0),
]

# Validate all directed edges before making requests
for edge_id in selected_edges:
    assert G.has_edge(*edge_id), f"Missing edge: {edge_id}"

print("All three probe edges exist.")

# Exactly three API requests maximum
with TomTomClient() as client:
    snapshot_3, summary_3 = collect_traffic_snapshot(
        G,
        client,
        max_requests=3,
        selected_edge_ids=selected_edges,
        pause_s=0.5,
    )

print("\n========== COLLECTION SUMMARY ==========")
print("Requests:", summary_3["requests_made"])
print("Errors:", summary_3["request_errors"])
print("Matching:", summary_3["map_matching_diagnostics"])
print("API errors:", summary_3["api_error_groups"])

print("\n========== EDGE RESULTS ==========")

for edge_id in selected_edges:
    u, v, k = edge_id
    edge = snapshot_3[u][v][k]

    print("\nEdge:", edge_id)
    print("Matched:", edge["traffic_matched"])
    print("Match reason:", edge["traffic_match_reason"])
    print("Traffic status:", edge["traffic_status"])
    print("Current speed:", edge["current_speed_kph"])
    print("Free flow speed:", edge["free_flow_speed_kph"])
    print("Confidence:", edge["traffic_confidence"])
    print("Imputed:", edge["time_is_imputed"])


All three probe edges exist.

========== COLLECTION SUMMARY ==========
Requests: 3
Errors: 0
Matching: {'candidate_match': 3}
API errors: []

========== EDGE RESULTS ==========

Edge: (1356901661, 109869, 0)
Matched: True
Match reason: candidate_match
Traffic status: clear
Current speed: 27.0
Free flow speed: 27.0
Confidence: 1.0
Imputed: False

Edge: (26389375, 34519894, 0)
Matched: True
Match reason: candidate_match
Traffic status: clear
Current speed: 23.0
Free flow speed: 23.0
Confidence: 1.0
Imputed: False

Edge: (7902276163, 25291778, 0)
Matched: True
Match reason: candidate_match
Traffic status: clear
Current speed: 18.0
Free flow speed: 18.0
Confidence: 1.0
Imputed: False


In [16]:
import inspect

from routing.traffic_pipeline import collect_traffic_snapshot

source = inspect.getsource(collect_traffic_snapshot)

print(
    "Reverse matching available:",
    "_reverse_edge_ids" in source
)

print(
    "Reverse reassignment available:",
    "reverse_direction_reassignments" in source
)

print("Summary keys:", sorted(summary_3.keys()))

Reverse matching available: True
Reverse reassignment available: True
Summary keys: ['api_error_groups', 'counts', 'directly_queried_directed_edges', 'fallback_speed_kph', 'finished_at_utc', 'geometry_and_heading_candidate_matches', 'heading_policy', 'map_matching_diagnostics', 'matched_coverage_percent', 'minimum_confidence', 'notes', 'probe_plan_count', 'reasons', 'request_errors', 'requests_made', 'reverse_candidate_checks', 'reverse_direction_reassignments', 'sampled_directed_edges', 'sampling_strategy', 'started_at_utc', 'stopped_http_status', 'total_directed_edges', 'usable_among_sampled_percent', 'usable_coverage_percent', 'usable_observed_edges']


In [17]:
tested_edges = {
    (1356901661, 109869, 0),
    (26389375, 34519894, 0),
    (7902276163, 25291778, 0),
}

# Previously failed probe: midpoint_too_far
known_failed_edges = {
    (109869, 265563774, 0),
}

excluded_edges = tested_edges | known_failed_edges

remaining_edges = [
    tuple(row["edge_id"])
    for row in plan
    if tuple(row["edge_id"]) not in excluded_edges
]

print("New probes:", len(remaining_edges))

for edge_id in remaining_edges:
    print(edge_id)

assert len(remaining_edges) > 0


New probes: 9
(1360237602, 3925964624, 0)
(1027237558, 1116949690, 0)
(1691189781, 20821126, 0)
(34519894, 6280192964, 0)
(20947178, 29353841, 0)
(26389442, 34519894, 0)
(1027237558, 860183693, 0)
(276548, 3764277424, 0)
(25291778, 7902276163, 0)


In [18]:
from routing.traffic_api import TomTomClient
from routing.traffic_pipeline import collect_traffic_snapshot

with TomTomClient() as client:

    snapshot_9, summary_9 = collect_traffic_snapshot(
        G,
        client,
        max_requests=len(remaining_edges),
        selected_edge_ids=remaining_edges,
        pause_s=0.5,
    )

print("========== COLLECTION SUMMARY ==========")

for key in [
    "requests_made",
    "request_errors",
    "directly_queried_directed_edges",
    "reverse_candidate_checks",
    "reverse_direction_reassignments",
    "geometry_and_heading_candidate_matches",
    "usable_observed_edges",
    "usable_coverage_percent",
    "usable_among_sampled_percent",
    "map_matching_diagnostics",
    "api_error_groups",
]:
    print(f"{key}: {summary_9.get(key)}")


========== COLLECTION SUMMARY ==========
requests_made: 9
request_errors: 3
directly_queried_directed_edges: 9
reverse_candidate_checks: 2
reverse_direction_reassignments: 2
geometry_and_heading_candidate_matches: 3
usable_observed_edges: 3
usable_coverage_percent: 0.05
usable_among_sampled_percent: 27.27
map_matching_diagnostics: {'heading_proxy_mismatch': 2, 'midpoint_too_far': 3, 'candidate_match': 1, 'reverse_candidate_match': 2}
api_error_groups: [{'http_status': 400, 'error_code': 'INVALID_REQUEST', 'count': 3, 'example_message_redacted': 'Point too far from nearest existing segment.'}]


In [20]:
import pandas as pd

rows = []

for u, v, k, d in snapshot_9.edges(
    keys=True,
    data=True
):
    if (
        d.get("traffic_directly_queried")
        or d.get("traffic_match_source") == "reverse_inferred"
    ):
        rows.append({
            "edge_id": (u, v, k),
            "highway": d.get("highway"),
            "match_source": d.get("traffic_match_source"),
            "match_reason": d.get("traffic_match_reason"),
            "status": d.get("traffic_status"),
            "current_speed": d.get("current_speed_kph"),
            "free_flow_speed": d.get("free_flow_speed_kph"),
            "confidence": d.get("traffic_confidence"),
            "imputed": d.get("time_is_imputed"),
            "reason": d.get("traffic_reason"),
        })

results_df = pd.DataFrame(rows)

print(results_df.to_string(index=False))


                    edge_id                       highway     match_source            match_reason  status  current_speed  free_flow_speed  confidence  imputed                       reason
    (276548, 3764277424, 0)                   residential              NaN  heading_proxy_mismatch missing            NaN              NaN         NaN     True      matched_on_reverse_edge
    (20947178, 29353841, 0)                   residential              NaN                     NaN missing            NaN              NaN         NaN     True                 api_http_400
  (25291778, 7902276163, 0)                       primary              NaN  heading_proxy_mismatch missing            NaN              NaN         NaN     True      matched_on_reverse_edge
    (26389442, 34519894, 0)                   residential              NaN        midpoint_too_far missing            NaN              NaN         NaN     True geometry_or_heading_mismatch
  (34519894, 6280192964, 0) [living_street, unclassifie

In [21]:
import sys
import routing
import routing.traffic_pipeline as tp

from pathlib import Path
import inspect

print("========== ENVIRONMENT ==========")

print("Python:", sys.executable)

print("Routing package:")
print(routing.__file__)

print("Traffic pipeline:")
print(tp.__file__)


print("\n========== SOURCE FILE CHECK ==========")

source_path = Path(tp.__file__)

source_on_disk = source_path.read_text(encoding="utf-8")

checks = [
    "_reverse_edge_ids",
    "reverse_direction_reassignments",
    "traffic_directly_queried",
    "reverse_candidate_match",
]

for item in checks:
    print(item, ":", item in source_on_disk)


print("\n========== LOADED FUNCTION CHECK ==========")

runtime_source = inspect.getsource(tp.collect_traffic_snapshot)

print(
    "Reverse helper in loaded function:",
    "_reverse_edge_ids" in runtime_source
)

print(
    "Reverse reassignment in loaded function:",
    "reverse_direction_reassignments" in runtime_source
)

========== ENVIRONMENT ==========
Python: /Users/mojtabashabani/wiser/mary/.venv/bin/python
Routing package:
/Users/mojtabashabani/wiser/AiWithPython/AiLeason/src/routing/__init__.py
Traffic pipeline:
/Users/mojtabashabani/wiser/AiWithPython/AiLeason/src/routing/traffic_pipeline.py

========== SOURCE FILE CHECK ==========
_reverse_edge_ids : True
reverse_direction_reassignments : True
traffic_directly_queried : True
reverse_candidate_match : True

========== LOADED FUNCTION CHECK ==========
Reverse helper in loaded function: True
Reverse reassignment in loaded function: True


In [22]:
import pandas as pd

plan_df = pd.DataFrame(plan)

print("========== ROAD CLASS DISTRIBUTION ==========")

print(
    plan_df["road_class"]
    .value_counts(dropna=False)
)

print("\n========== PROBE PLAN ==========")

display(
    plan_df.sort_values(
        ["road_class", "corridor_offset_m"]
    )
)


========== ROAD CLASS DISTRIBUTION ==========
road_class
residential     8
unclassified    2
trunk           1
primary         1
Name: count, dtype: int64

========== PROBE PLAN ==========


,edge_id,latitude,longitude,length_m,road_class,corridor,corridor_offset_m
11,"(25291778, 7902276163, 0)",51.494383,-0.174705,44.51,primary,South Kensington,42.6
2,"(1360237602, 3925964624, 0)",51.495273,-0.174187,68.82,residential,South Kensington,0.9
0,"(109869, 265563774, 0)",51.495550,-0.181906,134.42,residential,Gloucester Road,1.3
3,"(1027237558, 1116949690, 0)",51.496115,-0.180221,187.61,residential,Gloucester Road,18.2
8,"(26389442, 34519894, 0)",51.498196,-0.173748,41.05,residential,South Kensington,31.4
4,"(1691189781, 20821126, 0)",51.500769,-0.176459,72.93,residential,Royal Albert Hall,45.6
7,"(20947178, 29353841, 0)",51.500597,-0.178215,200.15,residential,Royal Albert Hall,49.2
9,"(1027237558, 860183693, 0)",51.496988,-0.179934,67.30,residential,Gloucester Road,49.3
10,"(276548, 3764277424, 0)",51.500031,-0.174850,56.28,residential,Royal Albert Hall,56.9
6,"(1356901661, 109869, 0)",51.495093,-0.182533,41.83,trunk,Gloucester Road,23.2


In [23]:
import networkx as nx
from collections import Counter

edges = list(
    snapshot_9.edges(keys=True, data=True)
)

sources = Counter()
statuses = Counter()

for u, v, k, data in edges:

    if data.get("traffic_matched"):
        sources[
            data.get("traffic_match_source", "unknown")
        ] += 1

        statuses[
            data.get("traffic_status", "unknown")
        ] += 1

print("========== OBSERVATION PROVENANCE ==========")
print("Match sources:", dict(sources))
print("Traffic statuses:", dict(statuses))

print(
    "Directly queried:",
    summary_9["directly_queried_directed_edges"]
)

print(
    "Reverse reassignments:",
    summary_9["reverse_direction_reassignments"]
)


========== OBSERVATION PROVENANCE ==========
Match sources: {'direct_probe': 1, 'reverse_inferred': 2}
Traffic statuses: {'clear': 3}
Directly queried: 9
Reverse reassignments: 2
